In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import ast     # TO CONVERT STRING INTO LIST
import re
import random
import string
import nltk
import nltk.corpus
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

import pickle

In [2]:
data_credits = pd.read_csv("tmdb_5000_credits.csv")
data_movies = pd.read_csv("tmdb_5000_movies.csv")

# Data Understanding

In [3]:
data_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
data_credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [5]:
movie = data_movies.merge(data_credits, on="title")

In [6]:
#movie.to_csv("movie.csv", index=False)

In [7]:
movie.sample(5)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
4053,0,"[{""id"": 53, ""name"": ""Thriller""}]",NaN,283384,"[{""id"": 14676, ""name"": ""series of murders""}]",en,The Calling,Detective Hazel Micallef hasn't had much to wo...,5.759493,"[{""name"": ""Darius Films"", ""id"": 5486}, {""name""...",...,108.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Pray for the Prey,The Calling,5.6,88,283384,"[{""cast_id"": 0, ""character"": ""Hazel Micallef"",...","[{""credit_id"": ""54e743ee9251416f5800134b"", ""de..."
1388,35000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",NaN,140823,"[{""id"": 5565, ""name"": ""biography""}, {""id"": 103...",en,Saving Mr. Banks,Author P.L. Travers travels from London to Hol...,31.957947,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,125.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Where her book ended, their story began.",Saving Mr. Banks,7.3,1226,140823,"[{""cast_id"": 13, ""character"": ""P.L. Travers"", ...","[{""credit_id"": ""52fe4a9d9251416c750e81cd"", ""de..."
222,115000000,"[{""id"": 878, ""name"": ""Science Fiction""}, {""id""...",NaN,68724,"[{""id"": 4565, ""name"": ""dystopia""}, {""id"": 1560...",en,Elysium,"In the year 2159, two classes of people exist:...",67.337670,"[{""name"": ""TriStar Pictures"", ""id"": 559}, {""na...",...,109.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,He can save us all.,Elysium,6.4,3439,68724,"[{""cast_id"": 3, ""character"": ""Max"", ""credit_id...","[{""credit_id"": ""52fe47a0c3a368484e0d17e7"", ""de..."
4527,2053648,"[{""id"": 99, ""name"": ""Documentary""}]",NaN,14271,"[{""id"": 5970, ""name"": ""wrestling""}, {""id"": 607...",en,Beyond the Mat,Beyond the Mat is a 1999 professional wrestlin...,2.778633,"[{""name"": ""Imagine Entertainment"", ""id"": 23}, ...",...,102.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Beyond the Mat,7.8,17,14271,"[{""cast_id"": 1, ""character"": ""Himself"", ""credi...","[{""credit_id"": ""58d5ec49c3a368127c03493c"", ""de..."
998,50000000,"[{""id"": 35, ""name"": ""Comedy""}]",http://www.zoolander.com/,329833,"[{""id"": 6241, ""name"": ""stupidity""}, {""id"": 966...",en,Zoolander 2,Derek and Hansel are modelling again when an o...,37.253774,"[{""name"": ""Scott Rudin Productions"", ""id"": 258...",...,100.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Long time no Z,Zoolander 2,4.7,797,329833,"[{""cast_id"": 0, ""character"": ""Derek Zoolander""...","[{""credit_id"": ""54ff0d5e9251410e56001a13"", ""de..."


In [8]:
movie.shape

(4809, 23)

In [9]:
movie.describe()

,budget,id,popularity,revenue,runtime,vote_average,vote_count,movie_id
count,4.809000e+03,4809.000000,4809.000000,4.809000e+03,4807.000000,4809.000000,4809.000000,4809.000000
mean,2.902780e+07,57120.571429,21.491664,8.227511e+07,106.882255,6.092514,690.331670,57120.571429
std,4.070473e+07,88653.369849,31.803366,1.628379e+08,22.602535,1.193989,1234.187111,88653.369849
min,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,5.000000
25%,7.800000e+05,9012.000000,4.667230,0.000000e+00,94.000000,5.600000,54.000000,9012.000000
50%,1.500000e+07,14624.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000,14624.000000
75%,4.000000e+07,58595.000000,28.350529,9.291317e+07,118.000000,6.800000,737.000000,58595.000000
max,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000,459488.000000


In [10]:
movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [11]:
movie.duplicated().sum()

0

# Data Cleaning

In [12]:
movie = movie[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]

In [13]:
movie.shape

(4809, 7)

In [14]:
movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4809 non-null   int64 
 1   title     4809 non-null   object
 2   overview  4806 non-null   object
 3   genres    4809 non-null   object
 4   keywords  4809 non-null   object
 5   cast      4809 non-null   object
 6   crew      4809 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.1+ KB


In [15]:
movie.isnull().mean()*100

movie_id    0.000000
title       0.000000
overview    0.062383
genres      0.000000
keywords    0.000000
cast        0.000000
crew        0.000000
dtype: float64

In [16]:
movie.dropna(inplace=True)

In [17]:
movie.duplicated().sum()

0

# Text Preprocessing

In [18]:
movie.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [19]:
movie.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [20]:
# TO CONVERT DICTIONARY'S  VALUES INTO REQUIRED LIST
def convert(object):
    l = []
    for i in ast.literal_eval(object):
        l.append(i["name"])
    return l

movie["genres"] = movie["genres"].apply(convert)

In [21]:
movie.iloc[0].genres

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [22]:
movie.iloc[0].keywords

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [23]:
movie["keywords"] = movie["keywords"].apply(convert)

In [24]:
movie.iloc[0].keywords

['culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d']

In [25]:
movie.iloc[0].cast

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [26]:
def convert2(object):
    l = []
    x = 0
    for i in ast.literal_eval(object):
        if x !=3:
            l.append(i["name"])
            x = x+1
        else:
            break
    return l

movie["cast"] = movie["cast"].apply(convert2)

In [27]:
movie.iloc[0].cast

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [28]:
#movie["crew"][0]
movie.iloc[0].crew

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [29]:
def find_director(object):
    l = []
    for i in ast.literal_eval(object):
        if i["job"] == "Director":
             l.append(i["name"])
             break
    return l

movie["crew"] = movie["crew"].apply(find_director)

In [30]:
movie.iloc[0].crew

['James Cameron']

In [31]:
movie.iloc[0].overview

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [32]:
movie["overview"] = movie["overview"].apply(lambda i: i.split())

In [33]:
movie.iloc[0].overview

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

In [34]:
movie.sample(15)

,movie_id,title,overview,genres,keywords,cast,crew
4232,127918,The Gatekeepers,"[In, an, unprecedented, and, candid, series, o...",[Documentary],"[israel, army, intelligence agency, shin bet]","[Ami Ayalon, Avraham Shalom, Yaakov Peri]",[Dror Moreh]
2765,16406,Dick,"[Comedy, about, two, high, school, girls, who,...",[Comedy],"[richard nixon, watergate, the white house]","[Kirsten Dunst, Michelle Williams, Will Ferrell]",[Andrew Fleming]
1393,2959,License to Wed,"[Newly, engaged,, Ben, and, Sadie, can't, wait...",[Comedy],"[new love, ten commandments, bride, bridegroom...","[Robin Williams, Mandy Moore, John Krasinski]",[Ken Kwapis]
4417,37694,Proud,"[The, true, story, of, the, only, African-Amer...",[Drama],[woman director],"[Vernel Bagneris, Marcus Chait, Michael Ciesla]",[Mary Pat Kelly]
953,928,Gremlins 2: The New Batch,"[Young, sweethearts, Billy, and, Kate, move, t...","[Comedy, Horror, Fantasy]","[new york, monster, skyscraper, mutant, restau...","[Zach Galligan, Phoebe Cates, John Glover]",[Joe Dante]
1194,11362,The Count of Monte Cristo,"[Edmond, Dantés's, life, and, plans, to, marry...","[Action, Adventure, Drama, Thriller]","[loss of lover, lover (female), ex-lover, tort...","[Jim Caviezel, Guy Pearce, Richard Harris]",[Kevin Reynolds]
1589,533,The Curse of the Were-Rabbit,"[Cheese-loving, eccentric, Wallace, and, his, ...","[Adventure, Animation, Comedy, Family]","[competition, garden, vegetable, stop motion, ...","[Peter Sallis, Helena Bonham Carter, Ralph Fie...",[Nick Park]
1999,152601,Her,"[In, the, not, so, distant, future,, Theodore,...","[Romance, Science Fiction, Drama]","[artificial intelligence, computer, love, lone...","[Joaquin Phoenix, Scarlett Johansson, Rooney M...",[Spike Jonze]
4310,98369,Blue Like Jazz,"[A, young, man, must, find, his, own, way, as,...","[Drama, Comedy]",[book store],"[Marshall Allman, Claire Holt, Tania Raymonde]",[Steve Taylor]
4761,322745,Counting,"[An, associative, collection, of, visual, impr...",[Documentary],[],[],[Jem Cohen]


In [35]:
movie.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4806 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4806 non-null   int64 
 1   title     4806 non-null   object
 2   overview  4806 non-null   object
 3   genres    4806 non-null   object
 4   keywords  4806 non-null   object
 5   cast      4806 non-null   object
 6   crew      4806 non-null   object
dtypes: int64(1), object(6)
memory usage: 300.4+ KB


In [36]:
# TO REMOVE THE SPACES WITHIN THE WORDS

movie["genres"] = movie["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
movie["keywords"] = movie["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movie["cast"] = movie["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movie["crew"] = movie["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [37]:
movie.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [38]:
movie["tags"] = movie["overview"] + movie["genres"] + movie["keywords"] + movie["cast"] + movie["crew"]

In [39]:
movie["tags"][1]

['Captain',
 'Barbossa,',
 'long',
 'believed',
 'to',
 'be',
 'dead,',
 'has',
 'come',
 'back',
 'to',
 'life',
 'and',
 'is',
 'headed',
 'to',
 'the',
 'edge',
 'of',
 'the',
 'Earth',
 'with',
 'Will',
 'Turner',
 'and',
 'Elizabeth',
 'Swann.',
 'But',
 'nothing',
 'is',
 'quite',
 'as',
 'it',
 'seems.',
 'Adventure',
 'Fantasy',
 'Action',
 'ocean',
 'drugabuse',
 'exoticisland',
 'eastindiatradingcompany',
 "loveofone'slife",
 'traitor',
 'shipwreck',
 'strongwoman',
 'ship',
 'alliance',
 'calypso',
 'afterlife',
 'fighter',
 'pirate',
 'swashbuckler',
 'aftercreditsstinger',
 'JohnnyDepp',
 'OrlandoBloom',
 'KeiraKnightley',
 'GoreVerbinski']

In [40]:
movie.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [41]:
movie_data = movie[["movie_id", "title", "tags"]]
movie_data.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [42]:
# CONVERTING LIST TO STRING & CONVERTING TO LOWERCASE

movie_data["tags"] = movie_data["tags"].apply(lambda  i: " ".join(i))
movie_data["tags"] = movie_data["tags"].apply(lambda  i: i.lower())

/tmp/ipykernel_17091/1583622536.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_data["tags"] = movie_data["tags"].apply(lambda  i: " ".join(i))
/tmp/ipykernel_17091/1583622536.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_data["tags"] = movie_data["tags"].apply(lambda  i: i.lower())


In [43]:
movie_data["tags"][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [44]:
movie_data.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [45]:
#REMOVING LINKS, TOKENISATION AND REMOVING PUNCTUATIONS

def preprocessing1(text):
    te = re.sub(r'http\S+', "", text)
    tokens = word_tokenize(te)
    
    punctuations = string.punctuation
    txt = []
    for i in tokens:
        if i not in punctuations:
            txt.append(i)
    return txt

#REMOVING STOPWORDS & CONVERTING TO LOWERCASE

def preprocessing2(text):
    stopword = stopwords.words("english")
    ntxt = []
    for j in text:
        if j not in stopword:
            ntxt.append(j.lower())
    return ntxt

#STEMMING

def preprocessing3(text):
    stem = []
    Stemmer = PorterStemmer()
    for i in text:
        stem.append(Stemmer.stem(i))
    return " ".join(stem)

In [46]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/mohit/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [47]:
movie_data["tags"] = movie_data["tags"].apply(preprocessing1)
movie_data["tags"] = movie_data["tags"].apply(preprocessing2)
movie_data["tags"] = movie_data["tags"].apply(preprocessing3)

/tmp/ipykernel_17091/3516406077.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_data["tags"] = movie_data["tags"].apply(preprocessing1)
/tmp/ipykernel_17091/3516406077.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_data["tags"] = movie_data["tags"].apply(preprocessing2)
/tmp/ipykernel_17091/3516406077.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentat

In [48]:
movie_data["tags"][0]

'22nd centuri parapleg marin dispatch moon pandora uniqu mission becom torn follow order protect alien civil action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [49]:
movie_data.head()

,movie_id,title,tags
0,19995,Avatar,22nd centuri parapleg marin dispatch moon pand...
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believ dead come back li...
2,206647,Spectre,cryptic messag bond ’ past send trail uncov si...
3,49026,The Dark Knight Rises,follow death district attorney harvey dent bat...
4,49529,John Carter,john carter war-weari former militari captain ...


# Creating User Interest Data

In [50]:
users = pd.read_csv("users.csv")
users.head()

,user_id,user_name,email
0,1,Mohit,mohit07@gmail.com
1,2,Amitesh,amitesh03@gmail.com
2,3,Virat,virat18@gmail.com
3,4,Roman,roman01@gmail.com


In [51]:
movie.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [52]:
# Display the unique genres

unique_genres = set([genre for genres_list in movie['genres'] for genre in genres_list])
print(unique_genres)

{'Adventure', 'Family', 'Horror', 'History', 'Animation', 'Documentary', 'ScienceFiction', 'Action', 'Music', 'Western', 'TVMovie', 'War', 'Fantasy', 'Drama', 'Foreign', 'Crime', 'Romance', 'Comedy', 'Thriller', 'Mystery'}


In [53]:
user_interests = {
    1: ['Mystery', 'Comedy', 'Romance', 'ScienceFiction', 'Drama'],  # Mohit
    2: ['War', 'Crime', 'Action', 'Thriller', 'Music'],              # Amitesh
    3: ['Horror', 'Animation', 'Adventure', 'Mystery', 'Action'],    # Virat
    4: ['Documentary', 'Western', 'Family', 'Comedy', 'Romance']     # Roman
}

In [54]:
# Parameters
num_users = 4
num_movies = 4806
num_interactions = 256  # Desired number of interactions

# Helper function to get movies with the most genre matches
def get_best_matching_movies(user_id, movies):
    interests = user_interests[user_id]
    
    # Calculate the number of matching genres for each movie
    movies['genre_match_count'] = movies['genres'].apply(
        lambda genres: len(set(genres).intersection(set(interests)))
    )
    
    # Filter movies with at least 2 matching genre and sort by match count
    matching_movies = movies[movies['genre_match_count'] > 2].sort_values(by='genre_match_count', ascending=False)
    
    # Return the list of movie IDs sorted by the highest genre match count
    return matching_movies['movie_id'].tolist()

# Generate Interactions
interactions = []

for _ in range(num_interactions):
    user_id = random.choice(users['user_id'])
    # Get the best matching movies for the user
    best_matching_movie_ids = get_best_matching_movies(user_id, movie)
    if not best_matching_movie_ids:
        continue  # Skip if no matching movies found
    movie_id = random.choice(best_matching_movie_ids)
    rating = random.choice([1, 2, 3, 4, 5])  # Random rating from 1 to 5

    # Add interaction
    interactions.append({
        'user_id': user_id,
        'movie_id': movie_id,
        'rating': rating
    })

# Create Interaction DataFrame
interaction_df = pd.DataFrame(interactions)

# Optionally, reset the index
interaction_df.reset_index(drop=True, inplace=True)

In [55]:
interaction_df.sample(15)

,user_id,movie_id,rating
16,2,336004,2
29,4,268171,2
94,2,72387,3
122,3,23685,5
239,2,49526,3
230,1,5178,4
224,1,5994,3
74,2,34314,3
245,1,22913,4
209,3,82702,5


In [56]:
interaction_df.shape

(256, 3)

In [57]:
merged_df = pd.merge(interaction_df, movie, on='movie_id', how='left')

In [58]:
user_1_interactions = merged_df[merged_df['user_id'] == 1]
print(user_1_interactions.shape)
user_1_interactions.sample(20)

(72, 11)


,user_id,movie_id,rating,title,overview,genres,keywords,cast,crew,tags,genre_match_count
15,1,62764,1,Mirror Mirror,"[After, she, spends, all, her, money,, an, evi...","[Adventure, Fantasy, Drama, Comedy, ScienceFic...","[attemptedmurder, fairytale, blackmagic, cockr...","[JuliaRoberts, LilyCollins, ArmieHammer]",[TarsemSingh],"[After, she, spends, all, her, money,, an, evi...",2
133,1,9621,2,Elizabethtown,"[Drew, Baylor, is, fired, after, causing, his,...","[Comedy, Drama, Romance]","[suicide, hotelroom, suicideattempt, newlove, ...","[OrlandoBloom, KirstenDunst, SusanSarandon]",[CameronCrowe],"[Drew, Baylor, is, fired, after, causing, his,...",2
151,1,217708,3,Of Horses and Men,"[A, country, romance, about, the, human, strea...","[Drama, Romance, Comedy]","[horse, snowstorm, icelandic]","[IngvarEggertSigurðsson, CharlotteBøving, Stei...",[BenediktErlingsson],"[A, country, romance, about, the, human, strea...",2
119,1,53256,2,Three,"[Hanna, and, Simon, are, in, a, 20, year, marr...","[Romance, Drama, Comedy]","[sex, bisexual, science]","[SophieRois, SebastianSchipper, DevidStriesow]",[TomTykwer],"[Hanna, and, Simon, are, in, a, 20, year, marr...",2
88,1,9583,5,Divine Secrets of the Ya-Ya Sisterhood,"[A, mother, and, daughter, dispute, is, resolv...","[Comedy, Drama, Romance]","[secretsociety, conciliation, marriage, mother...","[SandraBullock, EllenBurstyn, FionnulaFlanagan]",[CallieKhouri],"[A, mother, and, daughter, dispute, is, resolv...",2
27,1,25113,2,The Four Seasons,"[Three, middle-aged, wealthy, couples, take, v...","[Comedy, Drama, Romance]",[],"[AlanAlda, CarolBurnett, LenCariou]",[AlanAlda],"[Three, middle-aged, wealthy, couples, take, v...",2
18,1,47607,1,Tiny Furniture,"[After, graduating, from, film, school,, Aura,...","[Romance, Comedy, Drama]","[sistersisterrelationship, malefemalerelations...","[LenaDunham, LaurieSimmons, GraceDunham]",[LenaDunham],"[After, graduating, from, film, school,, Aura,...",2
112,1,14709,5,Varsity Blues,"[In, small-town, Texas,, high, school, footbal...","[Comedy, Drama, Romance]","[americanfootball, smalltown, texas, cheerlead...","[JamesVanDerBeek, AmySmart, JonVoight]",[BrianRobbins],"[In, small-town, Texas,, high, school, footbal...",2
165,1,14474,2,The Oh in Ohio,"[Priscilla, and, Jack, appear, to, be, the, pe...","[Comedy, Drama, Romance]","[sex, adultery, depression, infidelity, nightc...","[ParkerPosey, DannyDeVito, PaulRudd]",[BillyKent],"[Priscilla, and, Jack, appear, to, be, the, pe...",2
166,1,222936,4,Aloha,"[A, celebrated, military, contractor, returns,...","[Drama, Comedy, Romance]","[lovetriangle, hawaii, satellite, military, du...","[BradleyCooper, EmmaStone, RachelMcAdams]",[CameronCrowe],"[A, celebrated, military, contractor, returns,...",2


In [59]:
# mohit -> mystery, comedy, drama, romance, ScienceFiction
# amitesh -> war, crime, action, thriller, music
# virat -> horror, animation, Adventure, mystery, action
# roman -> documentary, Western, Family, comedy, romance

In [60]:
interaction_df.to_csv('movie_interaction_df.csv', index=False)

# Model Building

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle

In [62]:
users_interest_data = pd.read_csv('movie_interaction_df.csv')

In [63]:
users.head()

,user_id,user_name,email
0,1,Mohit,mohit07@gmail.com
1,2,Amitesh,amitesh03@gmail.com
2,3,Virat,virat18@gmail.com
3,4,Roman,roman01@gmail.com


In [64]:
users_interest_data.head()

,user_id,movie_id,rating
0,2,62835,5
1,1,508,2
2,1,10030,1
3,3,10329,5
4,1,45658,2


In [65]:
movie_data.head()

,movie_id,title,tags
0,19995,Avatar,22nd centuri parapleg marin dispatch moon pand...
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believ dead come back li...
2,206647,Spectre,cryptic messag bond ’ past send trail uncov si...
3,49026,The Dark Knight Rises,follow death district attorney harvey dent bat...
4,49529,John Carter,john carter war-weari former militari captain ...


In [66]:
movie_data.shape

(4806, 3)

In [67]:
cv = CountVectorizer(max_features=5000, stop_words="english")
tfd = TfidfVectorizer(max_features=3000)

In [68]:
vector = cv.fit_transform(movie_data["tags"]).toarray()
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [69]:
# FINDING THE DISTANCES BETWEEN THE MOVIES
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.08257228, 0.08385255, ..., 0.06282809, 0.02331262,
        0.        ],
       [0.08257228, 1.        , 0.06154575, ..., 0.02305715, 0.        ,
        0.        ],
       [0.08385255, 0.06154575, 1.        , ..., 0.04682929, 0.        ,
        0.        ],
       ...,
       [0.06282809, 0.02305715, 0.04682929, ..., 1.        , 0.05858749,
        0.04087596],
       [0.02331262, 0.        , 0.        , ..., 0.05858749, 1.        ,
        0.04550158],
       [0.        , 0.        , 0.        , ..., 0.04087596, 0.04550158,
        1.        ]])

In [70]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:6]

[(2405, 0.2592724864350674),
 (3728, 0.25391668753850405),
 (1214, 0.2514005711827466),
 (539, 0.24404676504598818),
 (507, 0.24293289905367482)]

In [71]:
# Calculate the average rating and number of interactions for each product
popularity_data = users_interest_data.groupby('movie_id').agg(
    average_rating=('rating', 'mean'),
    num_interactions=('movie_id', 'count')
).reset_index()

# Merge popularity data with the products data
movie_data = movie_data.merge(popularity_data, on='movie_id', how='left')

# Fill NaN values (if any) with 0
# Normalize the average_rating and num_interactions using MinMaxScaler
scaler = MinMaxScaler()
movie_data[['normalized_rating', 'normalized_interactions']] = scaler.fit_transform(
    movie_data[['average_rating', 'num_interactions']]
)

# Calculate a combined popularity score
movie_data['num_interactions'].fillna(0, inplace=True)

/tmp/ipykernel_17091/2785098508.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movie_data['num_interactions'].fillna(0, inplace=True)


In [72]:
# Step 1: Fill NaN values in the 'num_interactions' column with 0 (if there are any)
movie_data['num_interactions'].fillna(0, inplace=True)

# Step 2: Normalize 'average_rating' and 'num_interactions' using MinMaxScaler
scaler = MinMaxScaler()

# Normalize 'average_rating' and 'num_interactions' columns
movie_data[['normalized_rating', 'normalized_interactions']] = scaler.fit_transform(
    movie_data[['average_rating', 'num_interactions']]
)

# Step 3: Calculate the combined popularity score
movie_data['popularity_score'] = (movie_data['normalized_rating'] + movie_data['normalized_interactions']) / 2

# Step 4: Fill any NaN values in 'popularity_score' (if any)
movie_data['popularity_score'].fillna(0, inplace=True)

/tmp/ipykernel_17091/3306341541.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movie_data['num_interactions'].fillna(0, inplace=True)
/tmp/ipykernel_17091/3306341541.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [73]:
movie_data.head()

,movie_id,title,tags,average_rating,num_interactions,normalized_rating,normalized_interactions,popularity_score
0,19995,Avatar,22nd centuri parapleg marin dispatch moon pand...,NaN,0.0,NaN,0.0,0.0000
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believ dead come back li...,NaN,0.0,NaN,0.0,0.0000
2,206647,Spectre,cryptic messag bond ’ past send trail uncov si...,NaN,0.0,NaN,0.0,0.0000
3,49026,The Dark Knight Rises,follow death district attorney harvey dent bat...,4.5,2.0,0.875,0.4,0.6375
4,49529,John Carter,john carter war-weari former militari captain ...,NaN,0.0,NaN,0.0,0.0000


In [74]:
def recommend_movies_for_user(user_id, users_interest_data, movie_data, similarity_matrix, top_n=10, threshold=0.1):
    # Get the list of movie IDs the user is interested in (assuming interest is based on 'movie_id')
    user_interests = users_interest_data[users_interest_data['user_id'] == user_id]['movie_id'].tolist()
    # print(user_interests)

    # Check if the user has any interests (if the user is new and has no interests)
    if not user_interests:
        print("New user detected. Providing popularity-based recommendations.")
        
        # Sort movies based on popularity score or other metric (assuming popularity_score exists)
        popular_movies = movie_data.sort_values(by='popularity_score', ascending=False)
        
        # Return the top N popular movies
        return popular_movies['title'].head(top_n).tolist()

    # Get the indices of the user's interested movies in the movie_data DataFrame
    movie_indices = [movie_data[movie_data['movie_id'] == movie_id].index[0] for movie_id in user_interests]
    # print(user_interests)

    # Calculate the average similarity score for each movie based on the user's interests
    # similarity_matrix: Movie similarity scores between movies
    similarity_scores = sum(similarity_matrix[idx] for idx in movie_indices) / len(movie_indices)

    # Sort movies based on similarity scores (high to low)
    movie_list = sorted(list(enumerate(similarity_scores)), key=lambda x: x[1], reverse=True)
    # print(movie_list)
    

    # Filter out movies that the user has already interacted with and apply the threshold for similarity
    recommended_movies = [
        movie_data.iloc[i[0]]['title'] for i in movie_list
        if movie_data.iloc[i[0]]['movie_id'] not in user_interests and i[1] >threshold
    ]

    # Return the top N recommended movies
    print(f"Recommending {len(recommended_movies)} movies to user {user_id}")
    return recommended_movies


In [75]:
recommended_movies = recommend_movies_for_user(1, users_interest_data, movie_data, similarity)
print("Recommended Products:", recommended_movies)

Recommending 299 movies to user 1
Recommended Products: ['Boynton Beach Club', 'The Incredibly True Adventure of Two Girls In Love', 'Lovely & Amazing', 'Housefull', 'Griff the Invisible', 'Solitary Man', 'The House of Mirth', 'My Big Fat Greek Wedding 2', 'Man on the Moon', 'Quartet', 'Living Out Loud', 'Mystic Pizza', 'Love and Other Catastrophes', 'The Importance of Being Earnest', 'The Inkwell', 'Full Frontal', 'The Spectacular Now', 'Lovely, Still', 'My Last Day Without You', 'Four Single Fathers', "Jesus' Son", 'Peggy Sue Got Married', 'The Object of My Affection', 'Sweet Charity', 'Love Me Tender', 'Never Again', 'Life During Wartime', 'Code 46', 'Summer Catch', 'Hav Plenty', 'Love Letters', 'Mumford', 'Friends with Money', 'Bride & Prejudice', "The Time Traveler's Wife", 'Admission', 'The Brothers McMullen', 'Drumline', 'The Perfect Man', 'Safety Not Guaranteed', 'Tiny Furniture', "I Don't Know How She Does It", 'Woman on Top', 'Sliding Doors', 'The Heart of Me', 'Trust the Man

In [76]:
# mohit -> mystery, comedy, drama, romance, ScienceFiction
# amitesh -> war, crime, action, thriller, music
# virat -> horror, animation, Adventure, mystery, action
# roman -> documentary, Western, Family, comedy, romance

In [77]:
recommended_movie_details = movie[movie['title'].isin(recommended_movies)]
recommended_movie_details.sample(20)

,movie_id,title,overview,genres,keywords,cast,crew,tags,genre_match_count
2799,6615,Lars and the Real Girl,"[Sometimes, you, find, love, where, you'd, lea...","[Comedy, Romance, Drama]","[garage, lonewolf, dyinganddeath, loss, delusi...","[RyanGosling, EmilyMortimer, PaulSchneider]",[CraigGillespie],"[Sometimes, you, find, love, where, you'd, lea...",2
2416,2752,40 Days and 40 Nights,"[Matt, Sullivan's, last, big, relationship, en...","[Comedy, Romance]","[sexaddiction, laundromat]","[JoshHartnett, ShannynSossamon, AdamTrese]",[MichaelLehmann],"[Matt, Sullivan's, last, big, relationship, en...",2
2768,9683,Bubble Boy,"[Jimmy, is, young, man, who, was, born, withou...","[Adventure, Comedy, Drama, Romance]","[lovesickness, niagarafalls, crush, youth, ill...","[JakeGyllenhaal, SwoosieKurtz, MarleyShelton]",[BlairHayes],"[Jimmy, is, young, man, who, was, born, withou...",2
1121,509,Notting Hill,"[The, British, comedy, from, director, Roger, ...","[Romance, Comedy, Drama]","[londonengland, bookshop, birthday, newlove, f...","[JuliaRoberts, HughGrant, GinaMcKee]",[RogerMichell],"[The, British, comedy, from, director, Roger, ...",2
2474,13972,The Women,"[The, story, centers, on, a, group, of, gossip...","[Comedy, Drama, Romance]","[beautysalon, divorce, womandirector]","[ClorisLeachman, IndiaEnnenga, AnnetteBening]",[DianeEnglish],"[The, story, centers, on, a, group, of, gossip...",2
1474,228205,The Longest Ride,"[The, lives, of, a, young, couple, intertwine,...","[Drama, Romance]","[basedonnovel, artstudent, cowboy, injury, bul...","[ScottEastwood, BrittRobertson, LolitaDavidovich]","[GeorgeTillman,Jr.]","[The, lives, of, a, young, couple, intertwine,...",1
4609,52032,My Dog Tulip,"[The, story, of, a, man, who, rescues, a, Germ...","[Animation, Comedy, Drama]","[humananimalrelationship, dog, germanshepherd,...","[ChristopherPlummer, LynnRedgrave, IsabellaRos...",[PaulFierlinger],"[The, story, of, a, man, who, rescues, a, Germ...",1
438,6964,Something's Gotta Give,"[Harry, Sanborn, is, an, aged, music, industry...","[Drama, Comedy, Romance]","[agedifference, ladykiller, womandirector]","[JackNicholson, DianeKeaton, KeanuReeves]",[NancyMeyers],"[Harry, Sanborn, is, an, aged, music, industry...",2
2616,57943,The Love Letter,"[20th, century, computer, games, designer, Sco...","[Comedy, Drama, Fantasy, Romance]",[],"[CampbellScott, JenniferJasonLeigh, DavidDukes]",[DanCurtis],"[20th, century, computer, games, designer, Sco...",2
1626,53113,One True Thing,"[A, career, woman, reassesses, her, parents', ...","[Drama, Romance]","[dysfunctionalfamily, cancer, deathofparent]","[MerylStreep, RenéeZellweger, WilliamHurt]",[CarlFranklin],"[A, career, woman, reassesses, her, parents', ...",1


In [78]:
# Evaluation

In [79]:
def split_user_interests(users_interest_data, user_id):
    user_interests = users_interest_data[users_interest_data['user_id'] == user_id]['movie_id'].tolist()
    train_interests, test_interests = train_test_split(user_interests, test_size=0.3, random_state=42)
    return train_interests, test_interests

In [80]:
def evaluated_recommend_products_for_user(user_id, users_interest_data, products_data, similarity_matrix):
    # Split user interests into training and testing
    train_interests, test_interests = split_user_interests(users_interest_data, user_id)
    
    # Get the indices of these products in the products_data DataFrame
    product_indices = [products_data[products_data['movie_id'] == pid].index[0] for pid in train_interests]

    # Calculate the average similarity score for each product based on the user's training interests
    similarity_scores = sum(similarity_matrix[idx] for idx in product_indices) / len(product_indices)

    # Sort products based on similarity scores in descending order
    product_list = sorted(list(enumerate(similarity_scores)), key=lambda x: x[1], reverse=True)

    threshold = 0.1
    recommended_products = [
        products_data.iloc[i[0]]['movie_id'] for i in product_list
        if products_data.iloc[i[0]]['movie_id'] not in train_interests and i[1] > threshold
    ]

    # Return the top 15 recommended product IDs
    return recommended_products[:15], test_interests


In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_recommendations(recommended_products, test_interests):
    y_true = [1 if pid in test_interests else 0 for pid in recommended_products]
    y_pred = [1] * len(recommended_products)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-Score: {f1:.2f}")

In [82]:
recommended_movies, test_interests = evaluated_recommend_products_for_user(1, users_interest_data, movie_data, similarity)
evaluate_recommendations(recommended_movies, test_interests)

Precision: 0.07
Recall: 1.00
F1-Score: 0.12


In [83]:
# def recommend(movie):
#     movie_index = movie_data[movie_data["title"] == movie].index[0]     # TO GET THE INDEX VALUE OF THE SEARCHED MOVIE
#     distance = similarity[movie_index]
#     movie_list = sorted(list(enumerate(distance)), reverse=True, key=lambda x: x[1])     # TO GET THE FIRST 5 MOVIES SIMILAR TO SEARCHED ONE
    
#     for i in movie_list:
#          print(movie_data.iloc[i[0]].title)

# recommended_movies = recommend("The Dark Knight", movie_data, similarity, top_n=10)
# print("Recommended Movies:", recommended_movies)

In [84]:
pickle.dump(movie_data, open("movie_data.pkl","wb"))

In [85]:
pickle.dump(similarity, open("movie_similarity_matrix.pkl", "wb"))